# Generate and evaluate a benchmark

Use the named-model API to inspect a trajectory, then build independent tipping and null contexts. Each trajectory contributes one indicator trend score. Install from the repository root with `pip install -e .` before running these cells.

Windows, noise controls, score directions, and event definitions are explicit. The small batches below illustrate the API rather than estimate general predictive performance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tipbox.datasets import Microcosm, generate_dataset
from tipbox.indicators import rolling_ar1, rolling_variance
from tipbox.metrics import ensemble_roc, plot_roc
from tipbox.models import MayHarvesting, SaddleNode


## Inspect a harvesting trajectory

The upper fold is a forcing reference. The observed population can drop later. Noise multiplies the configured diffusion amplitude; `noise=1` retains that amplitude.

In [ ]:
model = MayHarvesting()
full = model.make_trajectory(
    n=1200,
    noise=1.0,
    seed=43,
)
reference = model.locate_transition(
    full.trajectory,
    null_simulation=False,
    param_values=full.param_values,
    time=full.time,
)

# Retain earlier observations before computing warning indicators.
time = np.asarray(full.time[:reference])
population = np.asarray(full.trajectory[:reference, 0])
ar1 = rolling_ar1(
    population,
    window=200,
    detrend="linear",
)
variance = rolling_variance(
    population,
    window=200,
    detrend="linear",
)
print(
    full.trajectory.shape,
    model.observable_names,
)


In [ ]:
fig, axes = plt.subplots(
    3,
    1,
    figsize=(9, 7),
    sharex=True,
)
axes[0].plot(
    full.time,
    full.trajectory[:, 0],
    color="0.75",
)
axes[0].plot(
    time,
    population,
    color="#216a91",
)
axes[0].set_ylabel("Population")
axes[1].plot(
    time,
    ar1,
    color="#216a91",
)
axes[1].set_ylabel("AR1 slope")
axes[2].plot(
    time,
    variance,
    color="#ad752d",
)
axes[2].set_ylabel("Variance")
axes[2].set_xlabel("Time (model units)")
for ax in axes:
    ax.axvline(
        float(full.time[reference]),
        color="#ba5c4b",
        linestyle="--",
    )
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Harvesting: indicators strictly before the fold reference")
fig.tight_layout()
plt.show()


## Generate fixed-length contexts

`n` sets the full simulation length. `num_timesteps` sets the retained context; `lead_steps=1` ends it one observation before its reference. Null rows precede tipping rows. Metadata records accepted seeds, rejection counts, and physical lead times.

In [ ]:
harvesting = generate_dataset(
    MayHarvesting(),
    num_timesteps=300,
    n_tipping=5,
    n_null=5,
    n=1000,
    noise=1.0,
    lead_steps=1,
    seed=43,
)
print(harvesting.observations.shape)  # (10, 300, 1)
print(harvesting.labels)
print(
    harvesting.metadata[-1]["seed"],
    harvesting.metadata[-1]["lead_time"],
)


## Compare two model mechanisms

The saddle-node model uses an absolute clock and a state-threshold reference. Its `noise` is a diffusion intensity; harvesting uses an amplitude multiplier. Equal context counts do not match physical durations or forcing histories. These outcome-conditioned batches do not estimate natural tipping probabilities.

In [ ]:
moving = generate_dataset(
    SaddleNode(epsilon=1.4),
    num_timesteps=300,
    n_tipping=5,
    n_null=5,
    noise=0.01,
    lead_steps=1,
    seed=43,
)


# Use the same declared analysis controls for both ensembles.
def warning_series(data):
    signals = data.observations[:, :, 0]
    return {
        "AR1": [
            rolling_ar1(
                x,
                window=100,
                detrend="linear",
            )
            for x in signals
        ],
        "Variance": [
            rolling_variance(
                x,
                window=100,
                detrend="linear",
            )
            for x in signals
        ],
    }


fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4),
)
for ax, title, data in zip(
    axes,
    ["Harvesting fold", "Moving equilibrium"],
    [harvesting, moving],
):
    report = ensemble_roc(
        data.labels,
        warning_series(data),
    )
    plot_roc(
        report,
        ax=ax,
    )
    ax.set_title(title)
    print(
        title,
        {
            name: (curve.auroc, curve.n_valid)
            for name, curve in report.curves.items()
        },
    )
fig.tight_layout()
plt.show()


## Inspect an observed recovery

The original Veraart et al. microcosm deposit contains both measured chemostats. It downloads into the user cache, preserving timestamps and missing readings. Data license: **CC BY-NC 4.0**. Cite [Veraart et al. (2012)](https://doi.org/10.1038/nature10723) and [DANS](https://doi.org/10.17026/dans-ztg-93aw).

The pinned M1 P1 analysis interval is rows `[161:411]`, following Grziwotz et al. (2023). Analyze each recovery separately; joining them would introduce false lagged pairs across perturbations. This observed case does not supply an independent binary control ensemble.

In [ ]:
record = Microcosm().load()
time = record.time[161:411]
signal = record.observations[161:411, 0]
variance = rolling_variance(
    signal,
    window=200,
    step=5,
    detrend="linear",
)
computed = np.isfinite(variance)
fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 3.5),
)
axes[0].plot(
    time - time[0],
    signal,
    color="#216a91",
)
axes[0].set(
    xlabel="Time within P1 recovery (day)",
    ylabel="Light attenuation (m⁻¹)",
)
axes[1].plot(
    (time - time[0])[computed],
    variance[computed],
    ".-",
    color="#ad752d",
)
axes[1].set(
    xlabel="Time within P1 recovery (day)",
    ylabel="Variance (m⁻²)",
)
fig.tight_layout()
plt.show()
print(int(computed.sum()))  # 11


The documentation explains [benchmark conventions](../docs/benchmarks.rst), [all empirical sources](../docs/models/index.rst#empirical-observations), and [figure reproduction](../docs/figures.rst).